# Проверка на своих картинках

Картинка целиком сжимается до 512×512, как на обучении. Модель даёт балл от 0 до 1 и карту 32×32, по клетке
на патч 16×16; карта растягивается обратно на картинку. Порог из `runs/pilot/report.md`, его пишет
`scripts/run_test.py`: выше порога около 1 % реальных картинок. Без отчёта порог 0.5.

In [ ]:
import os, re, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.model import Detector
from src.data import ImageDataset

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)
report = out / 'report.md'
threshold = float(re.search(r'threshold ([\d.]+)', report.read_text()).group(1)) if report.exists() else 0.5
threshold

In [ ]:
@torch.no_grad()
def show(path):
    """Картинка, балл и карта, где модель видит генерацию. Возвращает балл."""
    x = ImageDataset([{'path': str(path), 'label': 0}], cfg['train']['size'])[0]['image'][None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        patch_logits, image_logit = model(x)
    prob, heat = torch.sigmoid(image_logit.float()).item(), torch.sigmoid(patch_logits[0, 0].float()).cpu().numpy()

    image = Image.open(path).convert('RGB')
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))
    ax[0].imshow(image)
    ax[0].set_title(f"{Path(path).name}: {'сгенерирована' if prob > threshold else 'реальная'}, {prob:.3f}")
    ax[1].imshow(image)
    ax[1].imshow(heat, cmap='jet', alpha=0.45, vmin=0, vmax=1, extent=(0, image.width, image.height, 0), interpolation='bilinear')
    ax[1].set_title('где модель видит генерацию')
    for a in ax:
        a.axis('off')
    plt.show()
    return prob

In [ ]:
folder = Path('my_images')   # папка со своими картинками
scores = {p.name: show(p) for p in sorted(folder.iterdir()) if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}}
for name, prob in scores.items():
    print(f"{prob:.3f}  {'FAKE' if prob > threshold else 'real'}  {name}")